# AI-Based Hiring Prediction System
---
## End-to-End Machine Learning Project

This project builds a machine learning model to predict whether a


candidate will be hired or rejected based on resume data.

### Objective
To simulate an AI-based resume screening system used in HR analytics.

## Dataset Description

The dataset contains synthetic resume data with the following features:

- Skills
- Experience (Years)
- Education
- Certifications
- Job Role
- Salary Expectation
- Projects Count
- Recruiter Decision (Target Variable)

### Target Variable
- Hire → 1
- Reject → 0

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
print(os.listdir())

In [ ]:
df = pd.read_csv("AI-Based Hiring Prediction System.csv")

In [ ]:
import os
os.rename("AI-Based Hiring Prediction System.csv", "hiring_data.csv")

df = pd.read_csv("hiring_data.csv")

## Step 1: Import Libraries

We import essential libraries for data manipulation, preprocessing, and machine learning.

- Pandas & NumPy → Data handling
- re → Text cleaning
- Scikit-learn → ML models and preprocessing tools
- Pickle → Saving trained models for later use

## Step 2: Load Dataset

The dataset is loaded using Pandas.

This dataset contains resume-related information such as skills, experience, education, and recruiter decisions.

We use `read_csv()` to load the dataset into a DataFrame.

In [ ]:
print(df.shape)
print(df.columns)
df.head()

## Step 3: Data Cleaning

We remove unnecessary columns:
- Resume_ID
- Name

These do not contribute to prediction.

We also convert the target variable:
- Hire → 1
- Reject → 0

This is required because ML models work with numerical values.

In [ ]:
# Drop unwanted columns
df.drop(["Resume_ID", "Name"], axis=1, inplace=True)

# Convert target
df["Recruiter Decision"] = df["Recruiter Decision"].map({
    "Hire": 1,
    "Reject": 0
})

## Step 4: Handling Missing Values

Missing values can cause errors during model training.

We handle them by:
- Replacing missing text values with empty strings
- Replacing missing numeric values with 0

This ensures smooth preprocessing and avoids runtime errors.

In [ ]:
df["Skills"] = df["Skills"].fillna("")
df["Certifications"] = df["Certifications"].fillna("")
df["Job Role"] = df["Job Role"].fillna("")

df["Experience (Years)"] = df["Experience (Years)"].fillna(0)
df["Salary Expectation ($)"] = df["Salary Expectation ($)"].fillna(0)
df["Projects Count"] = df["Projects Count"].fillna(0)

## Step 5: Text Cleaning

We combine text features:
- Skills
- Certifications
- Job Role

into a single column: `combined_text`.

Cleaning steps:
- Convert text to lowercase
- Remove special characters
- Remove extra spaces

This ensures consistency for text processing.

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["combined_text"] = (
    df["Skills"] + " " +
    df["Certifications"] + " " +
    df["Job Role"]
)

df["combined_text"] = df["combined_text"].apply(clean_text)

## Step 6: Text Vectorization (TF-IDF)

Machine learning models cannot understand raw text.

We use TF-IDF (Term Frequency - Inverse Document Frequency) to convert text into numerical features.

TF-IDF assigns higher importance to meaningful words and lower importance to common words.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=500)
X_text = tfidf.fit_transform(df["combined_text"]).toarray()

## Step 7: Encoding Categorical Data

The Education column is categorical.

We use Label Encoding to convert it into numerical values.

This allows the model to process it as input.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["Education"] = le.fit_transform(df["Education"].astype(str))

## Step 8: Feature Engineering

We combine:
- Text features (TF-IDF output)
- Numeric features (experience, salary, projects, education)

This creates the final feature matrix used for training the model.

In [ ]:
import numpy as np

numeric = df[[
    "Experience (Years)",
    "Salary Expectation ($)",
    "Projects Count",
    "Education"
]].values

X = np.concatenate((X_text, numeric), axis=1)
y = df["Recruiter Decision"].values

## Step 9: Train-Test Split

We split the dataset into:
- 80% Training Data
- 20% Testing Data

This helps evaluate how well the model performs on unseen data.

It also helps prevent overfitting.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Step 10: Feature Scaling

We apply StandardScaler to numeric features.

Scaling ensures all features are on a similar range, improving model performance.

Note: Only numeric features are scaled, not TF-IDF features.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train[:, -4:] = scaler.fit_transform(X_train[:, -4:])
X_test[:, -4:] = scaler.transform(X_test[:, -4:])

## Step 11: Model Training

We use Random Forest Classifier.

Why Random Forest?
- Handles both numerical and text-based features well
- Reduces overfitting
- Provides good accuracy for structured datasets

The model is trained using the training data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

## Step 12: Model Evaluation

We evaluate the model using:
- Accuracy Score → overall correctness
- Classification Report → precision, recall, F1-score

This helps us understand model performance in detail.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

## Step 13: Saving the Model

We save the trained model and preprocessing objects using Pickle.

This allows us to reuse the model without retraining.

In [ ]:
import pickle

pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(le, open("encoder.pkl", "wb"))

## Step 14: Prediction System

We create a function that:
- Takes user input
- Applies preprocessing
- Predicts hiring decision

This simulates a real-world AI resume screening system.

In [ ]:
def predict_hiring(skills, exp, edu, certs, job_role, projects, salary):

    text = skills + " " + certs + " " + job_role
    text = clean_text(text)
    text_vec = tfidf.transform([text]).toarray()

    if edu in le.classes_:
        edu_encoded = le.transform([edu])[0]
    else:
        edu_encoded = 0

    num = np.array([[exp, salary, projects, edu_encoded]])
    num_scaled = scaler.transform(num)

    final_input = np.concatenate((text_vec, num_scaled), axis=1)

    pred = model.predict(final_input)[0]
    prob = model.predict_proba(final_input)[0][1]

    return ("Hire" if pred == 1 else "Reject", prob)

## Step 15: Testing the Model

We test the prediction system using sample input to verify end-to-end functionality.

In [ ]:
result = predict_hiring(
    skills="Python Machine Learning SQL",
    exp=3,
    edu="B.Tech",
    certs="AWS Certified",
    job_role="Data Scientist",
    projects=5,
    salary=60000
)

print("Prediction:", result)

## Final Conclusion

- Built a complete end-to-end ML pipeline
- Performed data cleaning and preprocessing
- Applied TF-IDF for text feature extraction
- Used Random Forest for classification
- Achieved good accuracy on test data
- Developed a prediction system simulating real-world hiring

### Learning Outcome
This project demonstrates how AI can automate resume screening in HR systems.